# Naming-game conventions in LLM populations

*Week 5 companion for the 🟫 Ashery, Aiello & Baronchelli (2025) discussion.*

The original paper (*Science Advances* 11, eadu9368) populates a naming game with **24 LLM agents** (Llama-3-70B, Llama-3.1-70B, Claude-3.5-Sonnet, Llama-2-70B), runs **40 trials** of pairwise interaction, and reports three findings:

1. **Spontaneous emergence** of a global convention from purely local pairwise rewards (Fig. 1).
2. **Collective bias** — even when individual agents are statistically unbiased, the population systematically converges on certain names more than others (Fig. 2).
3. **Critical mass** — a committed minority of ∼2–67% (model-dependent) can flip an existing convention (Fig. 3).

The mechanism is genuinely simple: pair two agents, each picks a letter, both win +100 if they match and lose −50 if they don’t, each remembers the last $H$ interactions. The cleverness of the paper is in *what emerges from this minimal setup*, not in any architectural complexity. We can rebuild the whole simulation in a notebook.

Code companion to the paper: <https://github.com/Ariel-Flint-Ashery/AI-norms>.

### What we keep, what we drop

| Ashery et al. 2025 | This notebook |
|---|---|
| Pairwise naming game with payoff +100 / −50 | ✅ §1, §2 |
| Memory of last $H$ interactions | ✅ §2 |
| Spontaneous convention emergence (Fig. 1A) | ✅ §3 |
| Collective bias (Fig. 2A) | ✅ §4 |
| Committed minority + critical mass (Fig. 3) | ✅ §5 |
| 4 LLMs (Llama-2 / 3 / 3.1, Claude 3.5) compared | ❌ — we use one (`gpt-4o-mini`) |
| $N=24$ agents, 40 trials, pool size $W=10$ | ❌ — we run small ($N\le 6$, trials $\le 5$, $W=5$) for class time |
| Quantitative critical-mass thresholds | ❌ — we *demonstrate the flip*, not estimate the threshold precisely |

The compression is across **scale**, not mechanism. Every architectural piece in the paper is in this notebook; we just run smaller. Class-project extensions in §6 push back toward paper-scale runs.

**One pedagogical note up front.** The paper’s published numbers are model-dependent (Fig. 3B: critical mass = 2% for Llama-3-70B, 67% for Llama-2-70B-Chat — effectively no minority). With `gpt-4o-mini` you should *expect different numbers*. Treat that as the lesson, not a failure: the *mechanism* (emergence, bias, tipping) reproduces; the *quantitative thresholds* don’t transfer cleanly across models.

---

## 0. Setup

The notebook runs in two modes:

- **Live mode** — set `OPENAI_API_KEY` in the environment before launching Jupyter. All LLM calls hit the API and results are cached to `5_ashery_cache/llm_cache.json` so re-runs are free.
- **Cached mode** — no API key. Reads only from the shipped cache. Every cell still produces output for prompts covered by the cache.

Pre-running the notebook once with a key produces the cache the rest of the class reads from.

### How much does it cost to run?

End-to-end live cost: **roughly 1–2 US cents** at current `gpt-4o-mini` pricing ($0.15 / 1M input, $0.60 / 1M output, April 2026). The notebook makes on the order of 600 chat completions across §3 (population emergence), §4 (collective-bias replicates), and §5 (committed minority). With the shipped cache: $0. The cell at the bottom prints the actual cost of *your* run.

In [ ]:
import os

#os.environ["OPENAI_API_KEY"] = "sk-"

import json
import hashlib
import random
from pathlib import Path
from dataclasses import dataclass, field
from collections import deque, Counter
from typing import Optional

import numpy as np
import pandas as pd

CACHE_DIR = Path('5_ashery_cache')
CACHE_DIR.mkdir(exist_ok=True)
CACHE_FILE = CACHE_DIR / 'llm_cache.json'

if CACHE_FILE.exists():
    with open(CACHE_FILE) as f:
        _cache = json.load(f)
else:
    _cache = {}

api_key = os.environ.get('OPENAI_API_KEY')
if api_key:
    import openai
    client = openai.OpenAI(api_key=api_key)
    print(f'Live mode: OpenAI client ready. Cache has {len(_cache)} entries.')
else:
    client = None
    print(f'Cached mode: no OPENAI_API_KEY. Cache has {len(_cache)} entries.')


# --- token usage and pricing (April 2026 OpenAI rates, per 1M tokens) ---
PRICING_PER_TOKEN = {
    'gpt-4o-mini': {'in': 0.15 / 1e6, 'out': 0.60 / 1e6},
}
_usage = {'tokens': {}, 'calls': {'chat_live': 0, 'chat_cached': 0}}


def _bump(model, kind, n):
    _usage['tokens'][(model, kind)] = _usage['tokens'].get((model, kind), 0) + n


def _cache_key(kind, model, payload):
    h = hashlib.sha256(json.dumps([kind, model, payload], sort_keys=True).encode()).hexdigest()[:16]
    return f'{kind}:{model}:{h}'


def _save_cache():
    with open(CACHE_FILE, 'w') as f:
        json.dump(_cache, f, indent=2)


def llm(system, user, model='gpt-4o-mini', temperature=0.7, max_tokens=10):
    """Two-message chat completion. Cached by (model, system, user, temperature)."""
    key = _cache_key('chat', model, {'system': system, 'user': user, 'temperature': temperature})
    if key in _cache:
        _usage['calls']['chat_cached'] += 1
        return _cache[key]
    if client is None:
        raise RuntimeError(f'Prompt not in cache and no API key set:\n{user[:200]}...')
    r = client.chat.completions.create(
        model=model,
        temperature=temperature,
        max_tokens=max_tokens,
        messages=[
            {'role': 'system', 'content': system},
            {'role': 'user',   'content': user},
        ],
    )
    out = r.choices[0].message.content.strip()
    _bump(model, 'in',  r.usage.prompt_tokens)
    _bump(model, 'out', r.usage.completion_tokens)
    _usage['calls']['chat_live'] += 1
    _cache[key] = out
    _save_cache()
    return out


def print_cost_summary():
    print(f'API calls: {_usage["calls"]["chat_live"]} live + {_usage["calls"]["chat_cached"]} cached')
    print()
    if not _usage['tokens']:
        print('No live API calls made. Cost: $0.00 (entire notebook served from cache).')
        return
    total_cost = 0.0
    print(f'{"model":<20} {"in tok":>10} {"out tok":>10} {"cost (USD)":>12}')
    print('-' * 56)
    for model in sorted({m for m, _ in _usage['tokens']}):
        in_tok  = _usage['tokens'].get((model, 'in'),  0)
        out_tok = _usage['tokens'].get((model, 'out'), 0)
        rate = PRICING_PER_TOKEN.get(model, {'in': 0.0, 'out': 0.0})
        cost = in_tok * rate['in'] + out_tok * rate['out']
        total_cost += cost
        print(f'{model:<20} {in_tok:>10,} {out_tok:>10,} {"$" + format(cost, ".5f"):>12}')
    print('-' * 56)
    print(f'{"total":<20} {"":>10} {"":>10} {"$" + format(total_cost, ".5f"):>12}')


random.seed(42)
np.random.seed(42)

---

## 1. The naming-game prompt

Per the paper's *Materials and Methods* (p. 8), each agent receives a **system prompt** that fixes the game rules + payoff structure + the agent's objective, and a **user prompt** that supplies the memory of past rounds and asks for a choice. No examples, no chain-of-thought scaffolding — the paper is explicit (p. 2) that they want the model to choose without hints about *how* to decide.

The agent is positioned in the third person as an external observer asked to "predict a player's next action based on the history of the $H$ most recent interactions." This framing matters: the agents do **not** know the players' identities or personalities, only the action history.

In [ ]:
SYSTEM_PROMPT = """You are playing a coordination game with one other player.

In each round you both independently choose one name from the available pool.
  - If your choice MATCHES your partner's:  each player gains +100 points.
  - If your choices DIFFER:                   each player loses 50 points.

Your objective is to maximise your own accumulated point tally, conditional on the behaviour of your co-player.

The available name pool is: {pool}

Respond with EXACTLY one name from the pool. No explanation, no other text — just the name."""


USER_PROMPT_EMPTY = """No prior interactions yet. This is your first round.

Available pool: {pool}

Your choice:"""


USER_PROMPT_WITH_MEMORY = """Your memory of the last {H} rounds (most recent last):
{memory_block}

Accumulated score so far: {score}

Available pool: {pool}

Your choice:"""


def format_memory(memory):
    """memory: deque of (round, own_choice, partner_choice, payoff) tuples."""
    if not memory:
        return ''
    return '\n'.join(
        f"  round {r}: you chose {own!r}, partner chose {other!r} → {payoff:+d} points"
        for r, own, other, payoff in memory
    )

### 1.1 Demo — what does an empty-memory agent pick?

With no history at all, what does the LLM pick? The paper’s collective-bias finding (Fig. 2A) emerges from running this single call thousands of times across replicate trials. We do it once to see the format, then run it many times in §4 to see the bias.

In [ ]:
POOL = ['Q', 'M', 'X', 'F', 'J']   # paper uses W=10; we use 5 to keep cost down
H_DEFAULT = 5


def parse_choice(raw, pool):
    """Extract the first pool letter that appears in the response, robust to extra punctuation."""
    raw_up = raw.strip().upper()
    for tok in raw_up.replace(',', ' ').replace('.', ' ').split():
        tok = tok.strip("'\"")
        if tok in pool:
            return tok
    for ch in raw_up:
        if ch in pool:
            return ch
    return None


def pick_name(memory, pool, score=0, H=H_DEFAULT, temperature=0.7):
    sys = SYSTEM_PROMPT.format(pool=pool)
    if not memory:
        usr = USER_PROMPT_EMPTY.format(pool=pool)
    else:
        usr = USER_PROMPT_WITH_MEMORY.format(
            H=H, memory_block=format_memory(memory), score=score, pool=pool,
        )
    raw = llm(sys, usr, temperature=temperature, max_tokens=10)
    return parse_choice(raw, pool), raw


choice, raw = pick_name(memory=deque(), pool=POOL)
print(f'Empty-memory choice: {choice!r}   (raw response: {raw!r})')

---

## 2. Pairwise round, payoff, memory

We pair two agents, each calls `pick_name()` independently with their own memory, compute the payoff, and append the round to *each* agent’s memory. The paper sets memory length **H = 5** and payoffs **+100 / −50**.

This is the entire mechanism. Everything else in the paper — emergence, collective bias, critical mass — is what falls out when you scale this up.

In [ ]:
@dataclass
class Agent:
    aid: int
    pool: list
    H: int = H_DEFAULT
    memory: deque = field(default_factory=deque)
    score: int = 0
    committed_to: Optional[str] = None    # §5: a fixed strategy that overrides the LLM call

    def __post_init__(self):
        # Fix memory's maxlen to H so old rounds are evicted automatically
        if self.memory.maxlen != self.H:
            self.memory = deque(self.memory, maxlen=self.H)

    def choose(self, temperature=0.7):
        if self.committed_to is not None:
            return self.committed_to, f'(committed to {self.committed_to})'
        return pick_name(self.memory, self.pool, score=self.score, H=self.H,
                         temperature=temperature)


def pair_round(a, b, round_idx, payoff_match=100, payoff_miss=-50, temperature=0.7):
    """Run one round between two agents. Updates both agents' memory and score in place."""
    choice_a, _ = a.choose(temperature=temperature)
    choice_b, _ = b.choose(temperature=temperature)
    success = (choice_a is not None) and (choice_a == choice_b)
    payoff = payoff_match if success else payoff_miss
    # Each agent records the round from its own perspective
    a.memory.append((round_idx, choice_a, choice_b, payoff))
    b.memory.append((round_idx, choice_b, choice_a, payoff))
    a.score += payoff
    b.score += payoff
    return choice_a, choice_b, success

### 2.1 Demo — two agents converging

Two agents, 12 rounds. The first few rounds are typically a coin-flip; once one agent’s memory contains a successful match, it has a reason to repeat it, and the other agent quickly synchronises. The convention emerges between *just two agents*. §3 scales the same mechanism up to a population.

In [ ]:
random.seed(0)
a = Agent(aid=1, pool=POOL)
b = Agent(aid=2, pool=POOL)

print(f'{"round":>5}  {"agent 1":>8}  {"agent 2":>8}  {"match?":>7}  {"score 1":>8}  {"score 2":>8}')
print('-' * 58)
for r in range(12):
    ca, cb, ok = pair_round(a, b, round_idx=r)
    flag = '✓' if ok else '✗'
    print(f'{r:>5}  {str(ca):>8}  {str(cb):>8}  {flag:>7}  {a.score:>8}  {b.score:>8}')

---

## 3. Population emergence — Fig. 1A reproduction

Scale the same mechanism to **N** agents. Every round: pick a random pair, run `pair_round`, repeat. Track the **success rate** (fraction of recent rounds that ended in agreement) and the **modal-name share** (fraction of agents whose last choice matches the population mode) over time.

The paper’s Fig. 1A shows a sigmoidal convergence: low success at first because random pair-ups can’t entrench any name, then a sharp transition as one name picks up enough memory-mass across the population to win every encounter.

We use **N = 6, T = 30 rounds, W = 5, H = 5** here. The paper uses **N = 24, T ≈ 50, W = 10, H = 5**. Smaller N converges noisier; the qualitative shape should still appear.

In [ ]:
def simulate_population(N, T, pool, H=H_DEFAULT, temperature=0.7, seed=0):
    """Run a naming game on N agents for T rounds. Returns (successes, modal_share, agents)."""
    random.seed(seed)
    agents = [Agent(aid=i, pool=pool, H=H) for i in range(N)]
    successes = []
    modal_share = []
    for r in range(T):
        i, j = random.sample(range(N), 2)
        _, _, ok = pair_round(agents[i], agents[j], round_idx=r, temperature=temperature)
        successes.append(int(ok))
        last_choices = [ag.memory[-1][1] for ag in agents if ag.memory]
        if last_choices:
            modal_count = Counter(last_choices).most_common(1)[0][1]
            modal_share.append(modal_count / N)
        else:
            modal_share.append(0.0)
    return np.array(successes), np.array(modal_share), agents


N, T = 6, 30
successes, modal_share, agents = simulate_population(N=N, T=T, pool=POOL, seed=1)

import matplotlib.pyplot as plt
plt.rcParams['figure.dpi'] = 110

window = 5
rolling = pd.Series(successes).rolling(window, min_periods=1).mean()

fig, ax = plt.subplots(figsize=(7, 3.2))
ax.plot(rolling.index, rolling.values, color='tab:blue', lw=2,
        label=f'success rate (rolling window = {window})')
ax.plot(modal_share, color='tab:orange', lw=2, ls='--',
        label='modal-name share across N')
ax.set_xlabel('round')
ax.set_ylabel('rate')
ax.set_ylim(-0.05, 1.05)
ax.set_title(f'Convention emergence in N={N} agents, pool size W={len(POOL)}')
ax.legend(loc='lower right')
plt.tight_layout(); plt.show()

final_choices = [ag.memory[-1][1] for ag in agents if ag.memory]
print('\nFinal-round choice distribution:', Counter(final_choices).most_common())

---

## 4. Collective bias — Fig. 2A reproduction

The paper’s most striking finding lives here. *Which* name wins?

If the LLM were genuinely unbiased — every name in the pool equally likely from an empty memory — repeated independent trials would produce a uniform distribution of winning names. The paper’s Fig. 2A shows the opposite: certain names win consistently across trials, and **the bias is stronger than any individual-level bias measured on isolated agents.** Table 1 of the paper (p. 5) decomposes where the bias enters: not in interaction 1 (uniform), but accumulated across rounds 2 and 3 once memory configurations make some names “sticky”.

This is the **Schelling lesson** instantiated: unbiased agents, biased outcomes.

We run **5 independent trials** (paper uses 40). With $N = 6$ and $W = 5$, the result will be noisy — but the *non-uniformity* should still be visible against the uniform null (each name expected to win 1.0 of 5 trials).

In [ ]:
def winning_name(agents):
    last = [ag.memory[-1][1] for ag in agents if ag.memory]
    if not last:
        return None
    return Counter(last).most_common(1)[0][0]


N_TRIALS = 5
winners = []
for trial in range(N_TRIALS):
    _, _, agents = simulate_population(N=N, T=T, pool=POOL, seed=100 + trial)
    winners.append(winning_name(agents))

print('Winning name per trial:', winners)

counts = Counter(winners)
fig, ax = plt.subplots(figsize=(6, 3))
ax.bar(POOL, [counts.get(name, 0) for name in POOL], color='tab:purple')
ax.axhline(N_TRIALS / len(POOL), color='grey', ls='--', lw=1, label='uniform null')
ax.set_xlabel('name'); ax.set_ylabel('# trials won')
ax.set_title(f'Collective bias across {N_TRIALS} trials (uniform expectation: {N_TRIALS/len(POOL):.1f} per name)')
ax.legend()
plt.tight_layout(); plt.show()

---

## 5. Committed minority — Fig. 3 reproduction

Once a convention is established, can a small group of *adversarial* agents flip it?

The paper’s setup: pre-converge a population on convention “Q” (every agent’s memory is filled with successful Q-Q rounds). Then introduce **M committed agents** who *always* pick “M” regardless of payoff or memory. Run rounds. Watch whether the majority eventually flips from Q to M.

This is the **Centola et al. 2018** *Science* experiment, run in silico on LLMs. Centola’s human result: critical mass ≈ 25%. The Ashery paper’s Fig. 3B reports thresholds from 2% (Llama-3-70B) to 67% (Llama-2-70B-Chat) — *strongly* model-dependent.

We don’t sweep $M$ precisely (that’s a class-project extension). We pick a single $M$ that should plausibly flip the population in our setup ($M = 2$ of $N = 6$ ≈ 33%) and watch whether it does.

In [ ]:
def seed_converged(agents, name):
    """Pre-fill every agent's memory with H successful (name, name) interactions."""
    for ag in agents:
        for r in range(-ag.H, 0):
            ag.memory.append((r, name, name, 100))
        ag.score = 100 * ag.H


def simulate_with_minority(N, M_size, T, pool, majority_name, minority_name,
                           H=H_DEFAULT, seed=0):
    """Pre-converge on majority_name, mark the last M agents as committed to minority_name, run T rounds."""
    random.seed(seed)
    agents = [Agent(aid=i, pool=pool, H=H) for i in range(N)]
    seed_converged(agents, majority_name)
    for ag in agents[-M_size:]:
        ag.committed_to = minority_name
    minority_share = []
    for r in range(T):
        i, j = random.sample(range(N), 2)
        pair_round(agents[i], agents[j], round_idx=r)
        last = [ag.memory[-1][1] for ag in agents]
        minority_share.append(sum(1 for c in last if c == minority_name) / N)
    return np.array(minority_share), agents


N5, T5, M5 = 6, 50, 2
minority_share, agents5 = simulate_with_minority(
    N=N5, M_size=M5, T=T5, pool=POOL,
    majority_name='Q', minority_name='M', seed=7,
)

fig, ax = plt.subplots(figsize=(7, 3.2))
ax.plot(minority_share, color='tab:red', lw=2, label="share producing 'M'")
ax.axhline(M5 / N5, color='grey', ls=':', lw=1, label=f'M/N = {M5/N5:.2f} (committed share)')
ax.set_xlabel('round'); ax.set_ylabel("share producing 'M'")
ax.set_ylim(-0.05, 1.05)
ax.set_title(f"Committed-minority flip: M={M5} of N={N5}, pre-converged on 'Q'")
ax.legend(loc='center right')
plt.tight_layout(); plt.show()

print(f'\nFinal share producing the minority name: {minority_share[-1]:.2f}')

---

## 6. Group-project sparks

A few directions this notebook opens but doesn’t close. Each is a real, publishable question.

- **Sweep critical mass.** Run §5 for $M \in \{1, 2, 3, 4\}$ on $N = 12$; for each, average the final minority share over 5 trials. Reproduce Fig. 3B for `gpt-4o-mini`. Compare the threshold to the paper’s published Llama-3 / Claude / Llama-2 thresholds.

- **Cross-model comparison.** Swap the backbone for `gpt-4o`, `claude-3-5-sonnet`, or an open Llama on Groq. Are the bias and critical-mass results stable across models? (The paper’s Fig. 3B says they are not — that’s the whole point.)

- **Topology beyond random pairwise.** Replace the `random.sample(range(N), 2)` line with a fixed network: a ring, a small-world (Watts–Strogatz), a scale-free (Barabási–Albert). Does convention emergence change qualitatively? (See Centola 2010, *Science* — the network-vs-uniform comparison is its own paper.)

- **Memory length $H$.** The paper uses $H = 5$. What happens at $H = 1$ (Markovian, no real history)? At $H = 20$ (long memory)? Is convergence faster, slower, more biased?

- **Pool semantics.** Replace `['Q', 'M', 'X', 'F', 'J']` with words that have meaning: `['cat', 'dog', 'fish', 'bird', 'rabbit']`, or with names that have political valence. Does the bias direction track semantics? (Hypothesis: yes — and that gives you a way to measure LLM stereotyping mechanically.)

- **Asymmetric payoffs → norms with content.** The paper studies *conventions* (Lewis 1969 sense — pure coordination, all options equally good). Make the payoff matrix asymmetric (matching on ‘Q’ = +100, matching on ‘M’ = +50). Now coordination *and* content are at stake — closer to a Bicchieri-style **norm**. Does the population converge on the higher-payoff name, the bias-favoured name, or whichever it hits first?

- **Heterogeneous prompts.** Half the agents get the system prompt as written; half get a prompt framed as “you are negotiating with a partner” (no payoff numbers). Does the population still converge? Does collective bias survive heterogeneity?

- **Compare to humans.** Centola, Becker, Brackbill & Baronchelli 2018 (*Science* 360, ref [23] of the paper) ran the same naming game with human subjects on Mechanical Turk. Their critical-mass result is 25%. Read that paper alongside this notebook and ask: *which* aspects of LLM convention dynamics resemble human dynamics, and which are LLM-specific artefacts?

---

## 7. What did this run cost?

The cell below prints exact token counts and the dollar cost of *your* run. With the shipped cache, the cost is $0 — every prompt was a cache hit. If you ran live with your own key, the typical end-to-end cost is 1–2 US cents.

In [ ]:
print_cost_summary()